
# CV Markdown Rules (v1 — EN/FR)

---


In [1]:
#| default_exp core

In [2]:
#| hide
from nbdev.showdoc import *

In [3]:
#| export
example_markdown_fr='''# Informations personnelles
- Nom: Suvam Bhattacharya
- Titre: Data Scientist | Data Analytics | Scientific Computing
- Localisation: France
- Photo: ../static/photo.jpeg

# Résumé
Data scientist spécialisé en ingénierie des données et modélisation, avec une forte expérience en Python, SQL et environnements cloud. Conception de pipelines robustes et reproductibles, avec un accent sur l’automatisation, la scalabilité et la qualité des données.

# Contact
- Email: xxx@gmail.com
- Téléphone: +xxxx
- LinkedIn: https://www.linkedin.com/in/xxxxx/
- GitHub: https://github.com/xxxxx

# Compétences
## Langages
- Python
- SQL
- Bash
- MATLAB

## Cloud & DevOps
- AWS
- Terraform
- EKS
- CI/CD
- GitLab CI
- Slurm

## Data Engineering
- ETL (PostgreSQL, MySQL, Power BI, Tableau, Talend)
- Data Warehousing
- Modélisation des données
- Qualité des données

## Machine Learning
- Machine Learning
- ARIMA, SARIMA
- NLP, LLM
- Deeplearning




# Expérience
## Ingénieur Data / Cloud @ Eolen AS+
- Durée: 2024
- Localisation: Bruyères-le-Châtel
- Technologies: AWS, Terraform, EKS, Slurm, GitLab CI, Bash
- Détail: Mise en place de pipelines CI/CD pour le déploiement d’environnements de calcul distribués.
- Détail: Conception de workflows reproductibles pour le traitement de données à grande échelle.
- Détail: Automatisation de tâches critiques via scripts Bash pour améliorer la productivité des équipes.
- Détail: Optimisation de l’utilisation des ressources de calcul et de stockage sur infrastructure cloud.

## Data Scientist Freelance
- Durée: 2023
- Localisation:
- Technologies: ARIMA, SARIMA, Machine Learning, AWS
- Détail: Développement de modèles de prévision de rupture de stock basés sur des séries temporelles.
- Détail: Conception de pipelines de données et mise en place d’un data warehouse sur AWS.
- Détail: Analyse des résultats et contribution à la prise de décision stratégique.

## Enseignant en Informatique @ I.U.T.
- Durée: 2019 - 2021
- Localisation: Orsay
- Technologies: POO, algorithmes avancés
- Détail: Enseignement de la programmation orientée objet et des structures algorithmiques.
- Détail: Encadrement de projets étudiants et accompagnement pédagogique.

## Doctorant en Physique @ LPS
- Durée: 2018 - 2022
- Localisation: Orsay
- Technologies: Python, modélisation, simulation numérique
- Détail: Analyse de données scientifiques complexes en Python.
- Détail: Modélisation et simulation numérique de systèmes physiques à haute dimension.
- Détail: Développement d’outils d’analyse et d’interprétation des résultats expérimentaux.
- Détail: Thèse : https://theses.hal.science/tel-03577251/

## Stagiaire de recherche @ CNRS – Thales
- Durée: févr. 2018 - août 2018
- Localisation: Palaiseau
- Technologies: MATLAB, physique expérimentale
- Détail: Étude du gaz d'électrons bidimensionnel dans le groupe d'Albert Fert.
- Détail: Réalisation d’expériences et simulations numériques sous MATLAB.
- Détail: Contribution à une publication scientifique : https://advanced.onlinelibrary.wiley.com/doi/abs/10.1002/adma.202102102

# Projets
## Communauté IA & Mentorat
- Technologies: IA, mentoring, debugging, communauté
- Détail: Animation d’une communauté Discord dédiée à l’intelligence artificielle.
- Détail: Accompagnement d’étudiants en data science (debugging, compréhension des concepts).
- Détail: Transmission de bonnes pratiques en résolution de problèmes et structuration du raisonnement.

## Pratiques de pleine conscience & leadership
- Technologies: Effortless Mindfulness
- Détail: Facilitation de sessions EM orientées clarté mentale et prise de décision.
- Détail: Application des principes de présence et d’attention dans des contextes techniques et collaboratifs.

# Formation
## Diplôme d’ingénieur
- Institution: École Polytechnique
- Localisation: Palaiseau
- Année: 2014 - 2018

## Masters Data Management
- Institution: Datascientest
- Localisation: Paris
- Année: 2023

# Certifications
## AWS Certified Solutions Architect – Associate
- Émetteur: AWS
- Année:

## Applied Machine Learning in Python
- Émetteur: University of Michigan
- Année:

## Introduction to Data Science in Python
- Émetteur: University of Michigan
- Année:

## Machine Learning in Production
- Émetteur: DeepLearning.AI
- Année: '''

In [4]:
#| export
import re
from pathlib import Path
from jinja2 import Environment, FileSystemLoader


In [ ]:
def split_lines(markdown: str) -> list[str]:
    return markdown.replace("\r\n", "\n").replace("\r", "\n").split("\n")

In [5]:
split_lines(example_markdown_fr)[:10]

['# Informations personnelles',
 '- Nom: Suvam Bhattacharya',
 '- Titre: Data Scientist | Data Analytics | Scientific Computing',
 '- Localisation: France',
 '- Photo: ../static/photo.jpeg',
 '',
 '# Résumé',
 'Data scientist spécialisé en ingénierie des données et modélisation, avec une forte expérience en Python, SQL et environnements cloud. Conception de pipelines robustes et reproductibles, avec un accent sur l’automatisation, la scalabilité et la qualité des données.',
 '',
 '# Contact']

In [6]:
#| export
def parse_bullet_line(line: str) -> tuple[str | None, str]:
    """
    Parse a bullet line.

    Returns:
    - ('Nom', 'CV Special19') for: - Nom: CV Special19
    - (None, 'Python') for: - Python
    """
    stripped = line.strip()

    m = re.match(r"^-\s*([^:]+)\s*:\s*(.*)$", stripped)
    if m:
        return m.group(1).strip(), m.group(2).strip()

    m = re.match(r"^-\s*(.+)$", stripped)
    if m:
        return None, m.group(1).strip()

    raise ValueError(f"Line is not a bullet line: {line}")

In [7]:
#| export
def add_bullet_to_container(container: dict | list, key: str | None, value: str) -> dict | list:
    """
    Add a parsed bullet to either a dict or a list.

    Rules:
    - key:value bullets belong in dicts
    - plain bullets belong in lists
    - repeated dict keys become lists
    """
    if key is None:
        if isinstance(container, list):
            container.append(value)
            return container

        raise ValueError("Cannot add plain bullet item to dict container")

    if isinstance(container, dict):
        if key not in container:
            container[key] = value
        else:
            if not isinstance(container[key], list):
                container[key] = [container[key]]
            container[key].append(value)
        return container

    raise ValueError("Cannot add key:value bullet to list container")

In [8]:
#| export
def new_subsection_container(first_bullet_key: str | None) -> dict | list:
    """
    Decide subsection container type from the first bullet encountered.
    """
    if first_bullet_key is None:
        return []
    return {}

In [9]:
#| export
def markdown_to_nested_dict(markdown: str) -> dict:
    """
    Convert CV markdown into a nested dictionary using only its structure.

    Rules:
    - # Section -> top-level key
    - ## Subsection -> nested key under current section
    - Plain text directly under # Section -> stored as string
    - - key: value -> dictionary entry
    - - item -> list entry
    - repeated - key: value -> list of values
    - repeated ## subsection titles -> become a list
    """
    result = {}

    current_h1 = None
    current_h2 = None

    h1_container = None
    h2_container = None
    h1_text_lines = []

    def flush_h1_text():
        nonlocal h1_text_lines, h1_container
        if current_h1 is None or h1_container is None:
            h1_text_lines = []
            return

        text = "\n".join(line.strip() for line in h1_text_lines if line.strip()).strip()
        if text:
            result[current_h1] = text
        h1_text_lines = []

    def flush_h2():
        nonlocal current_h2, h2_container, h1_container

        if current_h1 is None or current_h2 is None or h2_container is None:
            return

        if current_h2 not in h1_container:
            h1_container[current_h2] = h2_container
        else:
            if not isinstance(h1_container[current_h2], list):
                h1_container[current_h2] = [h1_container[current_h2]]
            h1_container[current_h2].append(h2_container)

        current_h2 = None
        h2_container = None

    def flush_h1():
        nonlocal current_h1, h1_container
        if current_h1 is None:
            return

        if h1_container is not None and current_h1 not in result:
            result[current_h1] = h1_container

        current_h1 = None
        h1_container = None

    for line in split_lines(markdown):
        stripped = line.strip()

        if not stripped:
            continue

        h1_match = re.match(r"^#\s+(.*)$", stripped)
        if h1_match:
            flush_h2()
            flush_h1_text()
            flush_h1()

            current_h1 = h1_match.group(1).strip()
            h1_container = {}
            current_h2 = None
            h2_container = None
            continue

        h2_match = re.match(r"^##\s+(.*)$", stripped)
        if h2_match:
            flush_h2()

            current_h2 = h2_match.group(1).strip()
            h2_container = None
            continue

        if stripped.startswith("- "):
            key, value = parse_bullet_line(stripped)

            if current_h2 is not None:
                if h2_container is None:
                    h2_container = new_subsection_container(key)
                h2_container = add_bullet_to_container(h2_container, key, value)
            else:
                if h1_container is None:
                    h1_container = {}
                if key is None:
                    raise ValueError(
                        f"Plain bullet item found directly under section '{current_h1}', "
                        "but no subsection exists to hold a list."
                    )
                h1_container = add_bullet_to_container(h1_container, key, value)

            continue

        if current_h2 is None:
            h1_text_lines.append(stripped)
        else:
            raise ValueError(
                f"Unexpected free text inside subsection '{current_h2}': {stripped}"
            )

    flush_h2()
    flush_h1_text()
    flush_h1()

    return result

In [10]:
#| export
parsed_markdown = markdown_to_nested_dict(example_markdown_fr)
parsed_markdown

{'Informations personnelles': {'Nom': 'Suvam Bhattacharya',
  'Titre': 'Data Scientist | Data Analytics | Scientific Computing',
  'Localisation': 'France',
  'Photo': '../static/photo.jpeg'},
 'Résumé': 'Data scientist spécialisé en ingénierie des données et modélisation, avec une forte expérience en Python, SQL et environnements cloud. Conception de pipelines robustes et reproductibles, avec un accent sur l’automatisation, la scalabilité et la qualité des données.',
 'Contact': {'Email': 'xxx@gmail.com',
  'Téléphone': '+xxxx',
  'LinkedIn': 'https://www.linkedin.com/in/xxxxx/',
  'GitHub': 'https://github.com/xxxxx'},
 'Compétences': {'Langages': ['Python', 'SQL', 'Bash', 'MATLAB'],
  'Cloud & DevOps': ['AWS', 'Terraform', 'EKS', 'CI/CD', 'GitLab CI', 'Slurm'],
  'Data Engineering': ['ETL (PostgreSQL, MySQL, Power BI, Tableau, Talend)',
   'Data Warehousing',
   'Modélisation des données',
   'Qualité des données'],
  'Machine Learning': ['Machine Learning',
   'ARIMA, SARIMA',
   '

In [11]:
def render_cv_html(data: dict, template_name: str = "cv_fr.html", theme_css: str = "cv_fr.css") -> str:
    #templates_dir = Path("templates/")
    templates_dir = Path("..") / "templates"
    env = Environment(
        loader=FileSystemLoader(templates_dir),
        autoescape=True,
    )

    template = env.get_template(template_name)
    print(data,theme_css)
    html = template.render(data=data, theme_css=theme_css)
    return html

In [12]:
from weasyprint import HTML, CSS

def html_to_pdf(html: str, output_path: str = "../output/cv.pdf"):
    HTML(string=html).write_pdf(
    "cv.pdf",
    stylesheets=[CSS("../templates/cv_fr.css")]
    )


In [13]:
import os
print(os.getcwd())

/home/svm/PycharmProjects/PythonProject/NotebookCV/nbs


In [14]:
#html = render_cv_html(parsed_markdown)
#html_to_pdf(html)

In [15]:
print(parsed_markdown["Informations personnelles"]["Photo"])

from pathlib import Path

def fix_photo_path(data: dict) -> dict:
    photo = data.get("Informations personnelles", {}).get("Photo")

    if photo:
        abs_path = Path(photo).resolve()
        data["Informations personnelles"]["Photo"] = f"file://{abs_path}"

    return data
fix_photo_path(parsed_markdown)
print(parsed_markdown["Informations personnelles"]["Photo"])


../static/photo.jpeg
file:///home/svm/PycharmProjects/PythonProject/NotebookCV/static/photo.jpeg


In [16]:
from pathlib import Path

def fix_asset_paths(data: dict) -> dict:
    lk = Path("../static/linkedin-svgrepo-com.svg").resolve()
    gt = Path("../static/logo-github-svgrepo-com.svg").resolve()
    data["assets"] = {
        "linkedin_icon": f"file://{lk}",
        "github_icon": f"file://{gt}",
    }
    print(data["assets"])
    return data


In [17]:
fix_asset_paths(parsed_markdown)

{'linkedin_icon': 'file:///home/svm/PycharmProjects/PythonProject/NotebookCV/static/linkedin-svgrepo-com.svg', 'github_icon': 'file:///home/svm/PycharmProjects/PythonProject/NotebookCV/static/logo-github-svgrepo-com.svg'}


{'Informations personnelles': {'Nom': 'Suvam Bhattacharya',
  'Titre': 'Data Scientist | Data Analytics | Scientific Computing',
  'Localisation': 'France',
  'Photo': 'file:///home/svm/PycharmProjects/PythonProject/NotebookCV/static/photo.jpeg'},
 'Résumé': 'Data scientist spécialisé en ingénierie des données et modélisation, avec une forte expérience en Python, SQL et environnements cloud. Conception de pipelines robustes et reproductibles, avec un accent sur l’automatisation, la scalabilité et la qualité des données.',
 'Contact': {'Email': 'xxx@gmail.com',
  'Téléphone': '+xxxx',
  'LinkedIn': 'https://www.linkedin.com/in/xxxxx/',
  'GitHub': 'https://github.com/xxxxx'},
 'Compétences': {'Langages': ['Python', 'SQL', 'Bash', 'MATLAB'],
  'Cloud & DevOps': ['AWS', 'Terraform', 'EKS', 'CI/CD', 'GitLab CI', 'Slurm'],
  'Data Engineering': ['ETL (PostgreSQL, MySQL, Power BI, Tableau, Talend)',
   'Data Warehousing',
   'Modélisation des données',
   'Qualité des données'],
  'Machine L

In [18]:

html = render_cv_html(parsed_markdown)
html_to_pdf(html)

{'Informations personnelles': {'Nom': 'Suvam Bhattacharya', 'Titre': 'Data Scientist | Data Analytics | Scientific Computing', 'Localisation': 'France', 'Photo': 'file:///home/svm/PycharmProjects/PythonProject/NotebookCV/static/photo.jpeg'}, 'Résumé': 'Data scientist spécialisé en ingénierie des données et modélisation, avec une forte expérience en Python, SQL et environnements cloud. Conception de pipelines robustes et reproductibles, avec un accent sur l’automatisation, la scalabilité et la qualité des données.', 'Contact': {'Email': 'xxx@gmail.com', 'Téléphone': '+xxxx', 'LinkedIn': 'https://www.linkedin.com/in/xxxxx/', 'GitHub': 'https://github.com/xxxxx'}, 'Compétences': {'Langages': ['Python', 'SQL', 'Bash', 'MATLAB'], 'Cloud & DevOps': ['AWS', 'Terraform', 'EKS', 'CI/CD', 'GitLab CI', 'Slurm'], 'Data Engineering': ['ETL (PostgreSQL, MySQL, Power BI, Tableau, Talend)', 'Data Warehousing', 'Modélisation des données', 'Qualité des données'], 'Machine Learning': ['Machine Learning',

In [19]:
print("contact-social" in html)
Path("debug.html").write_text(html, encoding="utf-8")

False


18016

In [20]:
#| hide
import nbdev; nbdev.nbdev_export()